# Network Centrality + Vulnerability Analysis

Builds a graph from the OSM road network, computes degree/closeness/betweenness
centrality on road intersections, then runs a node-failure simulation: find the weak-link
junction on the user → nearest-gym route (the one whose closure forces the worst detour) and
measure the impact.

Coordinates are in metres (EPSG:32635 / UTM 35N), so edge weights are lengths in metres.

In [9]:
import arcpy
import networkx as nx
import os

# ! If you want to run the notebook: CHANGE TO YOUR absolute path
CURRENT_WORKING_DIR_PATH = r"C:\Users\HP ZBook 17 G5\Documents\ArcGIS\Projects\ModelingOfTransportationNetworksAndSystems_FinalProject"
GEODATABASE_PATH = os.path.join(CURRENT_WORKING_DIR_PATH, "ModelingOfTransportationNetworksAndSystems_FinalProject.gdb")

arcpy.env.workspace = GEODATABASE_PATH
arcpy.env.overwriteOutput = True

ROADS = "gis_osm_roads"
GYMS  = "gym_nodes_UTM35N"
USER  = "user_location_UTM35N"
ROUTE = "nearest_gym"

UTM_35N_CODE = 32635

print("Workspace:", arcpy.env.workspace)

Workspace: C:\Users\HP ZBook 17 G5\Documents\ArcGIS\Projects\ModelingOfTransportationNetworksAndSystems_FinalProject\ModelingOfTransportationNetworksAndSystems_FinalProject.gdb


## Build the graph from gis_osm_roads

Each polyline becomes one undirected edge between its first and last point, weighted by
shape.length (meters). Endpoints are rounded to whole metres so segments that share a
junction collapse to the same node.

In [10]:
roads_path = os.path.join(GEODATABASE_PATH, ROADS)
sr = arcpy.Describe(roads_path).spatialReference
print("Roads CRS:", sr.name, "(factoryCode =", sr.factoryCode, ")")

if sr.factoryCode != UTM_35N_CODE:
    graph_src = "memory\\roads_utm35n"
    print("Reprojecting roads to UTM 35N ->", graph_src)
    arcpy.management.Project(roads_path, graph_src, arcpy.SpatialReference(UTM_35N_CODE))
else:
    graph_src = roads_path
    print("Roads already in UTM 35N; using as-is.")

SNAP_M = 1  # rounding tolerance in metres for collapsing coincident endpoints
            # (increase to 2-5 if the node count looks far too high)

def node_key(point):
    return (round(point.X / SNAP_M) * SNAP_M, round(point.Y / SNAP_M) * SNAP_M)

G = nx.Graph()
with arcpy.da.SearchCursor(graph_src, ["SHAPE@", "OID@"]) as cursor:
    for shape, oid in cursor:
        if shape is None:
            continue
        a = node_key(shape.firstPoint)
        b = node_key(shape.lastPoint)
        if a == b:
            continue
        length = shape.length
        if G.has_edge(a, b) and G[a][b]["weight"] <= length:
            continue
        G.add_edge(a, b, weight=length)

print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())

Roads CRS: GCS_WGS_1984 (factoryCode = 4326 )
Reprojecting roads to UTM 35N -> memory\roads_utm35n
Nodes: 884826 | Edges: 598890


## Keep the largest connected component

OSM extracts contain stray disconnected fragments. Centrality and routing only make sense
on a single connected network.

In [11]:
n_before = G.number_of_nodes()
largest = max(nx.connected_components(G), key=len)
G = G.subgraph(largest).copy()

print(f"Largest component: {G.number_of_nodes()} nodes "
      f"({n_before - G.number_of_nodes()} dropped), {G.number_of_edges()} edges")

Largest component: 119297 nodes (765529 dropped), 129077 edges


## Compute the three centralities

- degree – local connectivity of a junction (instant).
- closeness – distance-weighted, but k-sampled. Exact closeness runs a Dijkstra from
  every node (119k here → hours), so we estimate each node's closeness from CLOSENESS_K
  random pivots: closeness(v) ≈ k / Σ d(pivot → v). Same scale as
  exact closeness, but approximate — note this on the poster.
- betweenness – the hero metric for finding chokepoints; k-sampled (BETWEENNESS_K,
  fixed seed), also approximate. Runs ~1–2 min silently (NetworkX prints nothing mid-run).

In [12]:
import random

COMPUTE_CLOSENESS = True
CLOSENESS_K   = min(500, G.number_of_nodes())   # pivots for sampled closeness
BETWEENNESS_K = min(500, G.number_of_nodes())   # samples for sampled betweenness

print("Computing degree centrality...")
degree = nx.degree_centrality(G)

if COMPUTE_CLOSENESS:
    print(f"Computing closeness centrality (sampled, {CLOSENESS_K} pivots)...")
    rng = random.Random(42)
    pivots = rng.sample(list(G.nodes), CLOSENESS_K)
    sum_dist = {v: 0.0 for v in G.nodes}
    reached  = {v: 0 for v in G.nodes}
    for i, p in enumerate(pivots, 1):
        for v, d in nx.single_source_dijkstra_path_length(G, p, weight="weight").items():
            sum_dist[v] += d
            reached[v]  += 1
        if i % 50 == 0:
            print(f"  ...{i}/{CLOSENESS_K} pivots")
    # Eppstein-Wang estimator: reciprocal of the mean distance to the sampled pivots
    closeness = {v: (reached[v] / sum_dist[v]) if sum_dist[v] > 0 else 0.0 for v in G.nodes}
    print("  closeness done.")
else:
    print("Skipping closeness (COMPUTE_CLOSENESS=False); writing zeros.")
    closeness = {n: 0.0 for n in G.nodes}

print(f"Computing betweenness centrality (sampled, {BETWEENNESS_K} samples)... ~1-2 min, no output")
betweenness = nx.betweenness_centrality(G, k=BETWEENNESS_K, weight="weight", seed=42)
print("All centralities done. Betweenness & closeness are sampled approximations.")

Computing degree centrality...
Computing closeness centrality (sampled, 500 pivots)...
  ...50/500 pivots
  ...100/500 pivots
  ...150/500 pivots
  ...200/500 pivots
  ...250/500 pivots
  ...300/500 pivots
  ...350/500 pivots
  ...400/500 pivots
  ...450/500 pivots
  ...500/500 pivots
  closeness done.
Computing betweenness centrality (sampled, 500 samples)... ~1-2 min, no output
All centralities done. Betweenness & closeness are sampled approximations.


## Write scores to a point feature class

road_nodes_centrality — one point per junction carrying all three scores. Symbolize with
graduated colors (closeness map, betweenness map).

In [13]:
out_fc_name = "road_nodes_centrality"
out_fc = os.path.join(GEODATABASE_PATH, out_fc_name)

if arcpy.Exists(out_fc):
    arcpy.management.Delete(out_fc)

arcpy.management.CreateFeatureclass(
    GEODATABASE_PATH, out_fc_name, "POINT",
    spatial_reference=arcpy.SpatialReference(UTM_35N_CODE)
)
for fld in ("degree", "closeness", "betweenness"):
    arcpy.management.AddField(out_fc, fld, "DOUBLE")

with arcpy.da.InsertCursor(out_fc, ["SHAPE@XY", "degree", "closeness", "betweenness"]) as cur:
    for n in G.nodes:
        cur.insertRow([n, degree.get(n, 0.0), closeness.get(n, 0.0), betweenness.get(n, 0.0)])

print("Wrote", G.number_of_nodes(), "points ->", out_fc)

Wrote 119297 points -> C:\Users\HP ZBook 17 G5\Documents\ArcGIS\Projects\ModelingOfTransportationNetworksAndSystems_FinalProject\ModelingOfTransportationNetworksAndSystems_FinalProject.gdb\road_nodes_centrality


## Locate the user and nearest-gym nodes on the graph

Snap the user point and the gym end of the existing nearest_gym route to their nearest
graph nodes. The route's two endpoints are classified by proximity to the user point.

In [14]:
def nearest_node(xy, graph):
    x0, y0 = xy
    best, best_d = None, None
    for nx_, ny_ in graph.nodes:
        d = (nx_ - x0) ** 2 + (ny_ - y0) ** 2
        if best_d is None or d < best_d:
            best, best_d = (nx_, ny_), d
    return best

# user coordinate
with arcpy.da.SearchCursor(os.path.join(GEODATABASE_PATH, USER), ["SHAPE@XY"]) as cur:
    user_xy = next(iter(cur))[0]

# the existing route's two endpoints: one is the user side, the other the gym side
with arcpy.da.SearchCursor(os.path.join(GEODATABASE_PATH, ROUTE), ["SHAPE@"]) as cur:
    route_shape = next(iter(cur))[0]
end_a = (route_shape.firstPoint.X, route_shape.firstPoint.Y)
end_b = (route_shape.lastPoint.X, route_shape.lastPoint.Y)

def dist2(p, q):
    return (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2

# whichever endpoint is closer to the user point is the user side; the other is the gym side
gym_xy = end_b if dist2(end_a, user_xy) <= dist2(end_b, user_xy) else end_a

user_node = nearest_node(user_xy, G)
gym_node  = nearest_node(gym_xy, G)
print("user_node:", user_node)
print("gym_node :", gym_node)

user_node: (199754, 4733405)
gym_node : (197081, 4731697)


## Vulnerability simulation — the key result

Find the baseline user → gym route, then identify the weak-link junction on that route and
close it. We test every interior junction of the route, remove it, and re-route — the one that
forces the longest detour is the trip's single point of failure.

This answers a concrete question: which one intersection, if it closed, hurts this trip the most,
and by how much? The earlier version removed the city-wide busiest junction, which sat ~36 km
from this 3.5 km route and so changed nothing (+0.0%). We also report that node's betweenness
rank, so the chokepoint map and this failure result reinforce each other on the poster.

In [15]:
# Baseline shortest path on the intact network
base_path = nx.shortest_path(G, user_node, gym_node, weight="weight")
base_len  = nx.shortest_path_length(G, user_node, gym_node, weight="weight")
print(f"Baseline route length: {base_len:,.1f} m  ({len(base_path)} junctions)")

# Worst-case single-junction failure ON this route:
# remove each interior junction in turn, re-route, and keep the one that hurts the most.
interior = base_path[1:-1]   # never remove the user or gym endpoint
print(f"Testing {len(interior)} interior junctions for the weak link...")

top_node, new_len = None, base_len
disconnects = []   # junctions whose closure cuts the gym off entirely

for node in interior:
    Gt = G.copy()
    Gt.remove_node(node)
    if nx.has_path(Gt, user_node, gym_node):
        cand_len = nx.shortest_path_length(Gt, user_node, gym_node, weight="weight")
        if cand_len > new_len:
            top_node, new_len = node, cand_len
    else:
        disconnects.append(node)

# If any closure fully disconnects the trip, that's the most severe failure — prefer it.
if disconnects:
    top_node = max(disconnects, key=lambda n: betweenness.get(n, 0.0))
    new_len = None
elif top_node is None:
    # No single removal lengthened the route (a parallel path of equal length exists).
    # Fall back to the most-central interior junction so we still report a meaningful node.
    top_node = max(interior, key=lambda n: betweenness.get(n, 0.0)) if interior else base_path[0]
    new_len = base_len
    print("  (route is robust: no single closure forces a detour; reporting most-central junction)")

# Betweenness rank of the chosen chokepoint (1 = busiest junction in the whole city)
ranked = sorted(betweenness, key=betweenness.get, reverse=True)
bw_rank = ranked.index(top_node) + 1 if top_node in betweenness else None

G2 = G.copy()
G2.remove_node(top_node)

print(f"\nWeak-link junction: {top_node}")
if bw_rank is not None:
    print(f"  citywide betweenness rank: #{bw_rank:,} of {len(ranked):,}  "
          f"(score={betweenness[top_node]:.4f})")

if new_len is not None and new_len > base_len:
    detour = (new_len - base_len) / base_len * 100
    print(f"  baseline:     {base_len:,.1f} m")
    print(f"  post-failure: {new_len:,.1f} m")
    print(f"  DETOUR: +{new_len - base_len:,.1f} m  ({detour:+.1f}%)")
elif new_len is None:
    print("  post-failure: NETWORK DISCONNECTED - gym unreachable if this junction closes.")
else:
    print(f"  post-failure: {new_len:,.1f} m  (no detour - trip is robust to single-junction failure)")

Baseline route length: 3,574.2 m  (44 junctions)
Testing 42 interior junctions for the weak link...

Weak-link junction: (199741, 4733359)
  citywide betweenness rank: #10,279 of 119,297  (score=0.0077)
  baseline:     3,574.2 m
  post-failure: 3,944.9 m
  DETOUR: +370.7 m  (+10.4%)


## Export routes + failed node for the failure map

route_before, route_after (polylines, simplified to straight segments between
junctions) and failed_node (the removed chokepoint).

In [16]:
def path_to_polyline(node_list):
    arr = arcpy.Array([arcpy.Point(x, y) for (x, y) in node_list])
    return arcpy.Polyline(arr, arcpy.SpatialReference(UTM_35N_CODE))

def write_polyline_fc(name, polyline):
    fc = os.path.join(GEODATABASE_PATH, name)
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)
    arcpy.management.CreateFeatureclass(
        GEODATABASE_PATH, name, "POLYLINE",
        spatial_reference=arcpy.SpatialReference(UTM_35N_CODE)
    )
    with arcpy.da.InsertCursor(fc, ["SHAPE@"]) as cur:
        cur.insertRow([polyline])
    return fc

# Baseline route
base_path = nx.shortest_path(G, user_node, gym_node, weight="weight")
print("route_before ->", write_polyline_fc("route_before", path_to_polyline(base_path)))

# Post-failure route, if one still exists
if new_len is not None:
    after_path = nx.shortest_path(G2, user_node, gym_node, weight="weight")
    print("route_after  ->", write_polyline_fc("route_after", path_to_polyline(after_path)))
else:
    print("route_after  -> skipped (network disconnected)")

# The failed junction as its own point FC
failed_fc_name = "failed_node"
failed_fc = os.path.join(GEODATABASE_PATH, failed_fc_name)
if arcpy.Exists(failed_fc):
    arcpy.management.Delete(failed_fc)
arcpy.management.CreateFeatureclass(
    GEODATABASE_PATH, failed_fc_name, "POINT",
    spatial_reference=arcpy.SpatialReference(UTM_35N_CODE)
)
with arcpy.da.InsertCursor(failed_fc, ["SHAPE@XY"]) as cur:
    cur.insertRow([top_node])
print("failed_node  ->", failed_fc)

route_before -> C:\Users\HP ZBook 17 G5\Documents\ArcGIS\Projects\ModelingOfTransportationNetworksAndSystems_FinalProject\ModelingOfTransportationNetworksAndSystems_FinalProject.gdb\route_before
route_after  -> C:\Users\HP ZBook 17 G5\Documents\ArcGIS\Projects\ModelingOfTransportationNetworksAndSystems_FinalProject\ModelingOfTransportationNetworksAndSystems_FinalProject.gdb\route_after
failed_node  -> C:\Users\HP ZBook 17 G5\Documents\ArcGIS\Projects\ModelingOfTransportationNetworksAndSystems_FinalProject\ModelingOfTransportationNetworksAndSystems_FinalProject.gdb\failed_node
